# 050 — MLP y backpropagation

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** Forward: z = 0.5·1 = 0.5; h = σ(0.5) = 0.6225; ŷ = 1·h = 0.6225;
L = ½·0.6225² = **0.1937**. Backward: ∂L/∂ŷ = ŷ−t = 0.6225.
∂L/∂w₂ = (ŷ−t)·h = 0.6225·0.6225 = **0.3875**.
∂L/∂h = (ŷ−t)·w₂ = 0.6225; σ'(z) = 0.6225·0.3775 = 0.2350;
δ = 0.6225·0.2350 = 0.1463; ∂L/∂w₁ = δ·x = **0.1463**.
Observa el patrón: cada gradiente es (señal que llega de arriba) × (derivada local).

**Ejercicio 2.** Con W₁=[[1,2],[3,4]] y W₂=[[0,1],[1,0]], W₂W₁=[[3,4],[1,2]].
Para cualquier x, aplicar W₁ y luego W₂ da lo mismo que aplicar W₂W₁ directamente:
la profundidad sin no linealidad es ilusoria.

**Ejercicio 3.** h₁ se activa si x₁+x₂ ≥ 0.5 (al menos un 1); h₂ si x₁+x₂ ≥ 1.5
(los dos son 1). ŷ = escalón(h₁−h₂−0.5) = "h₁ y no h₂" = "al menos uno pero no ambos"
= XOR. La primera capa transforma el espacio hasta hacer el problema separable.

**Ejercicio 4.** `evidence` contiene hechos inspeccionables de la corrida;
`limitations` impide extrapolar la demo a producción.


In [ ]:
result = run_lab("neural", seed=50)
assert result["kind"] == "neural"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math
sigmoid = lambda z: 1 / (1 + math.exp(-z))
step = lambda z: 1 if z >= 0 else 0

# Ejercicio 1
x, w1, w2, t = 1.0, 0.5, 1.0, 0.0
z = w1 * x; h = sigmoid(z); yhat = w2 * h; L = 0.5 * (yhat - t) ** 2
dL_dyhat = yhat - t
dL_dw2 = dL_dyhat * h
dL_dw1 = dL_dyhat * w2 * h * (1 - h) * x
print(f"h={h:.4f}  ŷ={yhat:.4f}  L={L:.4f}  ∂L/∂w2={dL_dw2:.4f}  ∂L/∂w1={dL_dw1:.4f}")
assert abs(dL_dw2 - 0.3875) < 1e-3 and abs(dL_dw1 - 0.1463) < 1e-3

# Ejercicio 3: XOR con 2 ocultas
def xor_net(x1, x2):
    h1 = step(x1 + x2 - 0.5)
    h2 = step(x1 + x2 - 1.5)
    return step(h1 - h2 - 0.5)
assert [xor_net(a, b) for a, b in [(0,0),(0,1),(1,0),(1,1)]] == [0, 1, 1, 0]
print("XOR verificado con una capa oculta de 2 neuronas")


## Reflexión

1. ¿Por qué el backward cuesta lo mismo (en orden) que el forward, y qué pasaría si calcularas cada ∂L/∂θ por diferencias finitas?
2. Si duplicas las neuronas ocultas de una red que ya resuelve XOR, ¿cambia la función que *puede* representar o la facilidad de *encontrarla*?
3. ¿Qué información del forward hay que guardar en memoria para poder hacer el backward, y qué implica eso para redes muy profundas?
